# Lab 01: Text Steganography using Zero-Width Character

**Course:** Information Hiding & Secret Sharing

**Topic:** Zero-Width Character Steganography

---

**Student Name:** Lê Thị Hồng Hạnh

**Student ID:** 22127103

---

## 1. Introduction

### 1.1 Background

Zero-width characters are special Unicode characters that have no visible representation but occupy space in a string. They are commonly used in text processing for purposes like controlling text flow, but they can also be exploited for steganography.

Since these characters are **completely invisible** to the human eye, they provide an excellent medium for hiding secret messages within innocent-looking text.

### 1.2 Zero-Width Characters

The four primary zero-width characters used in steganography:

| Character | Unicode | Name | Description |
|-----------|---------|------|-------------|
| ​ | U+200B | Zero-Width Space (ZWSP) | Invisible space, allows line breaks |
| ‌ | U+200C | Zero-Width Non-Joiner (ZWNJ) | Prevents ligature formation |
| ‍ | U+200D | Zero-Width Joiner (ZWJ) | Causes ligature formation |
| ﻿ | U+FEFF | Zero-Width No-Break Space (BOM) | Byte order mark, prevents line breaks |

### 1.3 Why Zero-Width Characters?

**Advantages:**
- Completely invisible in rendered text
- Survive copy-paste operations in most applications
- Work in emails, documents, web pages, social media
- High capacity (2 bits per character position)
- Text appears completely normal

**Disadvantages:**
- Easily detected by checking string length vs. visible length
- Can be stripped by some text processors
- Unicode-aware analysis will reveal them
- Some platforms filter zero-width characters

### 1.4 Encoding Scheme

We use 4 zero-width characters to encode 2 bits each:

| Bits | Character | Unicode | Python |
|------|-----------|---------|--------|
| 00 | Zero-Width Space | U+200B | `\u200b` |
| 01 | Zero-Width Non-Joiner | U+200C | `\u200c` |
| 10 | Zero-Width Joiner | U+200D | `\u200d` |
| 11 | Zero-Width No-Break Space | U+FEFF | `\ufeff` |

**Example:**
- Secret: "Hi" = `01001000 01101001` (binary)
- Split into 2-bit pairs: `01 00 10 00 01 10 10 01`
- Encoded: `\u200c\u200b\u200d\u200b\u200c\u200d\u200d\u200c`

---


## 2. Instructions

### How to Complete This Lab

⚠️ **Important:** This notebook will be graded using automated testing. Please follow the instructions carefully.

**Completing the Tasks**

1. Fill in your name and student ID at the top of this notebook.

2. Complete the code in cells marked with:
```python
# YOUR CODE HERE
raise NotImplementedError()
```

3. Delete the `raise NotImplementedError()` line when you implement your solution.

4. For optional tasks:
```python
# YOUR CODE HERE (OPTIONAL)
```

5. For written answers:
```markdown
YOUR ANSWER HERE
```

**Testing Your Code**

- Below each task, there are test cells to verify your implementation.
- If a test cell runs without errors, your code passes that test.
- ⚠️ Passing all visible tests does not guarantee full correctness - there may be hidden test cases.

**Before Submission**

1. Run `Kernel` → `Restart Kernel & Run All Cells` to ensure everything works.
2. Remove any debug print statements or extra cells you created.
3. **Do not modify** the provided code cells or test cells.

**Submission Format**

```
StudentID/
└── Part01/
    └── Lab01_Text_Steganography_Part1.ipynb
└── Part02/
    └── Lab01_Text_Steganography_Part2.ipynb
```

Compress the `StudentID` folder and submit.

---

**Academic Integrity**

🎯 The goal is to **learn authentically**. You may discuss ideas with classmates, but your code and answers must be your own work based on your genuine understanding.

🚫 **Plagiarism or cheating will result in a score of 0 for the entire course.**

---
## 3. Constants and Helper Functions

The following constants and helper functions are provided.

In [1]:
def text_to_bits(text: str) -> str:
    """
    Convert a text string to a binary string using UTF-8 encoding.
    
    Args:
        text: The text string to convert
    
    Returns:
        Binary string (e.g., "Hi" → "0100100001101001")
    
    Example:
        >>> text_to_bits('A')
        '01000001'
        >>> text_to_bits('Hi')
        '0100100001101001'
    """
    if not text:
        return ''
    
    # Encode text to UTF-8 bytes
    text_bytes = text.encode('utf-8')
    
    # Convert each byte to 8-bit binary string
    bits = ''.join(format(byte, '08b') for byte in text_bytes)
    
    return bits


def bits_to_text(bits: str) -> str:
    """
    Convert a binary string back to text using UTF-8 encoding.
    
    Args:
        bits: Binary string (must be valid UTF-8 when decoded)
    
    Returns:
        Decoded text string
    
    Example:
        >>> bits_to_text('01000001')
        'A'
        >>> bits_to_text('0100100001101001')
        'Hi'
    """
    bits = bits[:len(bits) // 8 * 8]
    if not bits:
        return ''
    try:        
        return bytes(int(bits[i:i+8], 2) for i in range(0, len(bits), 8)).decode('utf-8')
    except:
        return ''

In [2]:
# =============================================================================
# TEST: Helper Functions
# =============================================================================

# Test text_to_bits
assert text_to_bits('A') == '01000001', "text_to_bits('A') should be '01000001'"
assert text_to_bits('Hi') == '0100100001101001', "text_to_bits('Hi') should be '0100100001101001'"
assert text_to_bits('') == '', "text_to_bits('') should be ''"

# Test bits_to_text
assert bits_to_text('01000001') == 'A', "bits_to_text('01000001') should be 'A'"
assert bits_to_text('0100100001101001') == 'Hi', "bits_to_text should decode 'Hi'"
assert bits_to_text('') == '', "bits_to_text('') should be ''"

# Test roundtrip
test_strings = ['Hello', 'World!', '123', 'Test 123!@#']
for s in test_strings:
    assert bits_to_text(text_to_bits(s)) == s, f"Roundtrip failed for '{s}'"

# Test pad_bits
# assert pad_bits('101', 2) == '1010', "pad_bits('101', 2) should be '1010'"
# assert pad_bits('1010', 2) == '1010', "pad_bits('1010', 2) should be '1010'"
# assert pad_bits('1', 4) == '1000', "pad_bits('1', 4) should be '1000'"

print("✓ All helper function tests passed!")

✓ All helper function tests passed!


In [3]:
import random
# =============================================================================
# CONSTANTS: Zero-Width Character Mapping
# =============================================================================

# Zero-width characters used for encoding
ZW_CHARS = {
    '00': '\u200b',  # Zero-Width Space (ZWSP)
    '01': '\u200c',  # Zero-Width Non-Joiner (ZWNJ)
    '10': '\u200d',  # Zero-Width Joiner (ZWJ)
    '11': '\ufeff',  # Zero-Width No-Break Space (BOM/ZWNBSP)
}

# Reverse mapping for decoding
ZW_TO_BITS = {v: k for k, v in ZW_CHARS.items()}

# Set of all zero-width characters (for detection)
ZW_SET = set(ZW_CHARS.values())

print("Zero-Width Character Mapping:")
print("-" * 40)
for bits, char in ZW_CHARS.items():
    print(f"  '{bits}' → U+{ord(char):04X} (repr: {repr(char)})")

Zero-Width Character Mapping:
----------------------------------------
  '00' → U+200B (repr: '\u200b')
  '01' → U+200C (repr: '\u200c')
  '10' → U+200D (repr: '\u200d')
  '11' → U+FEFF (repr: '\ufeff')


---
## 4. Algorithm Overview


### 4.1 Encode to Zero-Width

```
FUNCTION encode_to_zw(secret_message):
    INPUT: secret_message (string) - Message to encode
    OUTPUT: zw_string (string) - Zero-width character string
    
    STEPS:
    1. Convert message to bits using text_to_bits()
    2. Create 16-bit length prefix (number of message bits)
    3. Concatenate: length_prefix + message_bits
    4. For each pair of bits:
       - Map to corresponding zero-width character:
         '00' → \u200b, '01' → \u200c, '10' → \u200d, '11' → \ufeff
    5. Return zero-width string
```

### 4.2 Decode from Zero-Width

```
FUNCTION decode_from_zw(zw_string):
    INPUT: zw_string (string) - Zero-width character string
    OUTPUT: secret_message (string) - Decoded message
    
    STEPS:
    1. For each zero-width character:
       - Map to corresponding bit pair:
         \u200b → '00', \u200c → '01', \u200d → '10', \ufeff → '11'
    2. Extract first 16 bits as length prefix
    3. Convert length prefix to integer
    4. Extract next 'length' bits as message bits
    5. Convert message bits to text using bits_to_text()
    6. Return decoded message
```

### 4.3 Embed in Cover Text

```
FUNCTION embed(cover_text, secret_message, strategy):
    INPUT:
        cover_text (string) - Original visible text
        secret_message (string) - Message to hide
        strategy (int) - Embedding strategy (1, 2, or 3)
    OUTPUT: stego_text (string) - Text with hidden message
    
    STEPS:
    1. Convert secret_message to zero-width string using encode_to_zw()
    2. Based on strategy:
       Strategy 1: Append zw_string to end of cover_text
       Strategy 2: Split cover_text into words
                   Distribute zw chars evenly after each word
       Strategy 3: Distribute zw chars after each character
    3. Return stego_text
```

### 4.4 Extract from Cover Text

```
FUNCTION extract(stego_text):
    INPUT: stego_text (string) - Text containing hidden message
    OUTPUT: secret_message (string) - Extracted message
    
    STEPS:
    1. Filter out all non-zero-width characters
    2. Decode zero-width string using decode_from_zw()
    3. Return secret message
```

---
### Example 1: Basic Encoding

**Input:** `"Hi"`

**Step-by-step:**
```
1. text_to_bits("Hi"):
   'H' = 72 = 01001000
   'i' = 105 = 01101001
   Result: "0100100001101001" (16 bits)

2. Length prefix (16 bits for value 16):
   16 = 0000000000010000

3. Combined: "0000000000010000" + "0100100001101001"
   = "00000000000100000100100001101001" (32 bits)

4. Split into pairs and encode:
   00 → \u200b (ZWSP)
   00 → \u200b
   00 → \u200b
   00 → \u200b
   00 → \u200b
   00 → \u200b
   01 → \u200c (ZWNJ)
   00 → \u200b
   00 → \u200b
   01 → \u200c
   00 → \u200b
   10 → \u200d (ZWJ)
   00 → \u200b
   01 → \u200c
   10 → \u200d
   01 → \u200c

Output: 16 zero-width characters (invisible)
```

### Example 2: Embedding with Strategy 1

**Input:**
- cover_text: `"Hello World"`
- secret_message: `"Hi"`
- strategy: `1`

**Output:** `"Hello World" + [16 ZW chars]` (looks like `"Hello World"`)

### Example 3: Embedding with Strategy 2

**Input:**
- cover_text: `"Hello World"`
- secret_message: `"Hi"`
- strategy: `2`

**Output:** `"Hello" + [8 ZW chars] + " World" + [8 ZW chars]`

### Example 4: Extraction

**Input:** Stego text from Example 2

**Step-by-step:**
```
1. Filter zero-width characters:
   Get 16 ZW chars

2. Decode each ZW char to bit pairs:
   \u200b → 00, \u200b → 00, ...
   Result: "00000000000100000100100001101001"

3. Extract length (first 16 bits):
   "0000000000010000" = 16

4. Extract message (next 16 bits):
   "0100100001101001"

5. Convert to text:
   01001000 = 'H'
   01101001 = 'i'

Output: "Hi"
```

**The implementation tasks (coding) in this lab contribute 4 points to your total score**

---
## 5. Task 1: Encode to Zero-Width

Implement the function to encode a secret message into zero-width characters.

In [4]:
# =============================================================================
# TASK 1: ENCODE TO ZERO-WIDTH
# =============================================================================

def encode_to_zw(secret_message: str) -> str:
    """
    Encode a secret message into a string of zero-width characters.
    
    Format: [16-bit length prefix][message bits][padding]
    - Length prefix: 16-bit binary representation of message bit count
    - Message bits: UTF-8 encoded message
    - Padding: Zeros to make total bits divisible by 2
    
    Each pair of bits is encoded as one zero-width character:
        '00' → \u200b (ZWSP)
        '01' → \u200c (ZWNJ)
        '10' → \u200d (ZWJ)
        '11' → \ufeff (ZWNBSP)
    
    Args:
        secret_message: The message to encode
    
    Returns:
        String of zero-width characters
    
    Example:
        >>> zw = encode_to_zw('Hi')
        >>> len(zw)  # 16 ZW characters for "Hi"
        16
    """
    # YOUR CODE HERE
    msg_bits = text_to_bits(secret_message)
    length_prefix = f'{len(msg_bits):0{16}b}'
    padding = '' if len(msg_bits) % 2 == 0 else '0'
    
    bits_string = length_prefix + msg_bits + padding
    zw_string = ''
    for i in range(0, len(bits_string), 2):
        zw_string += ZW_CHARS[bits_string[i:i+2]]
    return zw_string

In [5]:
# =============================================================================
# TEST: Encode to Zero-Width
# =============================================================================

# Test basic encoding
zw_hi = encode_to_zw('Hi')
assert len(zw_hi) == 16, f"encode_to_zw('Hi') should produce 16 ZW chars, got {len(zw_hi)}"
assert all(c in ZW_SET for c in zw_hi), "All characters should be zero-width"

# Test empty string
zw_empty = encode_to_zw('')
assert len(zw_empty) == 8, f"encode_to_zw('') should produce 8 ZW chars (length prefix only), got {len(zw_empty)}"

# Test single character
zw_a = encode_to_zw('A')
assert len(zw_a) == 12, f"encode_to_zw('A') should produce 12 ZW chars, got {len(zw_a)}"

# Test longer message
zw_hello = encode_to_zw('Hello')
expected_len = (16 + 5 * 8) // 2  # length prefix + 5 chars * 8 bits, divided by 2 bits per ZW char
assert len(zw_hello) == expected_len, f"encode_to_zw('Hello') should produce {expected_len} ZW chars"

print("✓ All encode_to_zw tests passed!")

✓ All encode_to_zw tests passed!


---
## 6. Task 2: Decode from Zero-Width

Implement the function to decode a secret message from zero-width characters.

In [6]:
# =============================================================================
# TASK 2: DECODE FROM ZERO-WIDTH
# =============================================================================

def decode_from_zw(zw_string: str) -> str:
    """
    Decode a secret message from a string of zero-width characters.
    
    Expects format: [16-bit length prefix][message bits][padding]
    
    Args:
        zw_string: String of zero-width characters
    
    Returns:
        Decoded secret message
    
    Example:
        >>> zw = encode_to_zw('Hi')
        >>> decode_from_zw(zw)
        'Hi'
    """
    # YOUR CODE HERE
    if not zw_string:
        return ''
    
    bits_string = ''
    for zw in zw_string:
        if zw not in ZW_SET:
            continue
        bits_string += ZW_TO_BITS[zw]
        
    if len(bits_string) < 16:
        return ''
    
    length = int(bits_string[:16], 2)
    if not length:
        return ''
    
    msg_bits = bits_string[16:16+length]
    decoded_msg = bits_to_text(msg_bits)
    
    return decoded_msg

In [7]:
# =============================================================================
# TEST: Decode from Zero-Width
# =============================================================================

# Test roundtrip for various messages
test_messages = [
    'Hi',
    'A',
    '',
    'Hello World!',
    'Test 123',
    'Special chars: @#$%',
]

for msg in test_messages:
    encoded = encode_to_zw(msg)
    decoded = decode_from_zw(encoded)
    assert decoded == msg, f"Roundtrip failed for '{msg}': got '{decoded}'"

print("✓ All decode_from_zw tests passed!")

✓ All decode_from_zw tests passed!


---
## 7. Task 3: Embed in Cover Text

Implement the function to embed a secret message in cover text using different strategies.

In [8]:
# =============================================================================
# TASK 3: EMBED IN COVER TEXT
# =============================================================================

def embed(cover_text: str, secret_message: str, strategy: int = 1) -> str:
    """
    Embed a secret message in cover text using zero-width characters.
    
    Strategies:
        1: Append all ZW characters at the end of cover text
        2: Distribute ZW characters after each word
        3: Distribute ZW characters after each visible character
    
    Args:
        cover_text: The visible text to hide the message in
        secret_message: The message to hide
        strategy: Embedding strategy (1, 2, or 3)
    
    Returns:
        Stego text containing the hidden message
    
    Raises:
        ValueError: If strategy is not 1, 2, or 3
        ValueError: If cover text doesn't have enough capacity for strategies 2 or 3
    
    Example:
        >>> stego = embed('Hello World', 'Hi', strategy=1)
        >>> stego[:11]  # Visible part unchanged
        'Hello World'
    """
    # YOUR CODE HERE
    stego_text = ''
    
    if strategy == 1:
        stego_text = cover_text + encode_to_zw(secret_message)
        
    elif strategy == 2:
        words = cover_text.split(' ')
        if not len(words):
            raise ValueError('Cover text has no words for strategy 2.')
            
        # ZWs per word
        zw_string = encode_to_zw(secret_message)
        zw_per_word = len(zw_string) // len(words)
        remaining_zw = len(zw_string) % len(words)
        # print(zw_per_word)
        # print(remaining_zw)
        
        zw_idx = 0 # Index of zw_string
        for i, word in enumerate(words):              
            end = zw_idx + zw_per_word + (1 if i < remaining_zw else 0)
            # print(i, end)
            stego_text += word + zw_string[zw_idx:end] + (' ' if i < len(words) - 1 else '')
            zw_idx = end
            
    elif strategy == 3:
        if not len(cover_text):
            raise ValueError('Cover text is empty for strategy 3.')
        
        # ZWs per char
        zw_string = encode_to_zw(secret_message)
        zw_per_char = len(zw_string) // len(cover_text)
        remaining_zw = len(zw_string) % len(cover_text)
            
        zw_idx = 0 # Index of zw_string
        for i, char in enumerate(cover_text):                
            end = zw_idx + zw_per_char + (1 if i < remaining_zw else 0)
            # print(i, end)
            stego_text += char + zw_string[zw_idx:end]
            zw_idx = end
        
    else:
        raise ValueError(f'Invalid strategy: {strategy}. Must be 1, 2, or 3.')
        
    return stego_text
    

In [9]:
# =============================================================================
# TEST: Embed in Cover Text
# =============================================================================

cover = "Hello World"
secret = "Hi"

# Test Strategy 1 (append at end)
stego1 = embed(cover, secret, strategy=1)
assert stego1.startswith(cover), "Strategy 1: Cover text should be at start"
assert len(stego1) > len(cover), "Strategy 1: Stego text should be longer"

# Test Strategy 2 (after each word)
stego2 = embed(cover, secret, strategy=2)
visible2 = ''.join(c for c in stego2 if c not in ZW_SET)
assert visible2 == cover, "Strategy 2: Visible text should match cover"

# Test Strategy 3 (after each character)
stego3 = embed(cover, secret, strategy=3)
visible3 = ''.join(c for c in stego3 if c not in ZW_SET)
assert visible3 == cover, "Strategy 3: Visible text should match cover"

# Test invalid strategy
try:
    embed(cover, secret, strategy=5)
    assert False, "Should raise ValueError for invalid strategy"
except ValueError:
    pass

print("✓ All embed tests passed!")

✓ All embed tests passed!


---
## 8. Task 4: Extract from Cover Text

Implement the function to extract a hidden message from stego text.

In [10]:
# =============================================================================
# TASK 4: EXTRACT FROM COVER TEXT
# =============================================================================

def extract(stego_text: str) -> str:
    """
    Extract a hidden message from stego text.
    
    This function works regardless of the embedding strategy used,
    as it simply collects all zero-width characters and decodes them.
    
    Args:
        stego_text: Text containing hidden message in zero-width characters
    
    Returns:
        Extracted secret message
    
    Example:
        >>> stego = embed('Hello World', 'Hi', strategy=1)
        >>> extract(stego)
        'Hi'
    """
    # YOUR CODE HERE
    zw_string = ''.join(c for c in stego_text if c in ZW_SET)
    extracted_msg = decode_from_zw(zw_string)
    return extracted_msg

In [11]:
# =============================================================================
# TEST: Extract from Cover Text
# =============================================================================

cover = "The quick brown fox jumps over the lazy dog"
test_secrets = ['Hi', 'Secret', 'Test 123!', '']

for secret in test_secrets:
    for strategy in [1, 2, 3]:
        stego = embed(cover, secret, strategy=strategy)
        extracted = extract(stego)
        assert extracted == secret, f"Extract failed for '{secret}' with strategy {strategy}: got '{extracted}'"

# Test extraction from text with no hidden message
plain_text = "No hidden message here"
extracted_plain = extract(plain_text)
assert extracted_plain == '', f"Extract from plain text should return empty string, got '{extracted_plain}'"

print("✓ All extract tests passed!")

✓ All extract tests passed!


---
## 9. Task 5: Detect Zero-Width Characters

Implement functions to detect and analyze zero-width characters in text (steganalysis).

In [12]:
# =============================================================================
# TASK 5: DETECT ZERO-WIDTH CHARACTERS
# =============================================================================

def detect_zero_width(text: str) -> dict:
    """
    Detect and analyze zero-width characters in text.
    
    This is a steganalysis function to detect potential hidden messages.
    
    Args:
        text: Text to analyze
    
    Returns:
        Dictionary with analysis results:
        {
            'has_zero_width': bool,      # True if any ZW chars found
            'total_zw_count': int,       # Total number of ZW chars
            'zw_positions': list,        # List of positions where ZW chars appear
            'zw_breakdown': dict,        # Count of each type of ZW char
            'estimated_capacity_bits': int,  # Estimated hidden data size in bits
            'suspicious': bool           # True if pattern suggests steganography
        }
    
    Example:
        >>> result = detect_zero_width('Hello\u200bWorld')
        >>> result['has_zero_width']
        True
        >>> result['total_zw_count']
        1
    """
    # YOUR CODE HERE
    zw_positions = []
    zw_breakdown = {}
    for i, c in enumerate(text):
        if c in ZW_SET:
            zw_positions.append(i)
            zw_breakdown[c] = zw_breakdown.get(c, 0) + 1
    
    total_zw_count = len(zw_positions)
    has_zero_width = total_zw_count > 0
    estimated_capacity_bits = total_zw_count * 2
    
    suspicious = False
    types_used = set(zw_breakdown.keys())
    
    if has_zero_width:
        # Heuristic 1: High count
        HIGH_COUNT_THRESHOLD = 8
        if total_zw_count >= HIGH_COUNT_THRESHOLD:
            suspicious = True
            
        # Heuristic 2: Multiple ZW types used together
        if len(types_used) > 2:
            suspicious = True
            
        # Heuristic 3: BOM found outside position 0
        BOM = '\ufeff'
        if BOM in types_used:
            bom_positions = [p for p in zw_positions if text[p] == BOM]
            if any(p != 0 for p in bom_positions):
                suspicious = True            
    
    return {
        'has_zero_width': has_zero_width,
        'total_zw_count': total_zw_count,
        'zw_positions': zw_positions,
        'zw_breakdown': zw_breakdown,
        'estimated_capacity_bits': estimated_capacity_bits,
        'suspicious': suspicious
    }
            

In [13]:
# =============================================================================
# TEST: Detect Zero-Width Characters
# =============================================================================

# Test with no zero-width characters
result_clean = detect_zero_width("Hello World")
assert result_clean['has_zero_width'] == False, "Clean text should not have ZW chars"
assert result_clean['total_zw_count'] == 0, "Clean text should have 0 ZW chars"

# Test with zero-width characters
stego = embed("Hello World", "Secret", strategy=1)
result_stego = detect_zero_width(stego)
assert result_stego['has_zero_width'] == True, "Stego text should have ZW chars"
assert result_stego['total_zw_count'] > 0, "Stego text should have positive ZW count"
assert result_stego['suspicious'] == True, "Stego text should be flagged as suspicious"

# Test positions
test_text = "A" + "\u200b" + "B" + "\u200c" + "C"
result_pos = detect_zero_width(test_text)
assert 1 in result_pos['zw_positions'], "Should detect ZW at position 1"
assert 3 in result_pos['zw_positions'], "Should detect ZW at position 3"

# Test breakdown
assert '\u200b' in result_pos['zw_breakdown'], "Should include ZWSP in breakdown"
assert '\u200c' in result_pos['zw_breakdown'], "Should include ZWNJ in breakdown"

print("✓ All detect_zero_width tests passed!")

✓ All detect_zero_width tests passed!


---
## 10. Task 6: Random Position Embedding

Implement embedding with random position distribution for increased security.

In [14]:
# =============================================================================
# TASK 6: RANDOM POSITION EMBEDDING
# =============================================================================

import random

def embed_random(cover_text: str, secret_message: str, seed: int) -> str:
    """
    Embed a secret message using random position distribution.
    
    This strategy distributes zero-width characters at random positions
    within the cover text, making detection more difficult.
    
    Args:
        cover_text: The cover text to hide the message in
        secret_message: The message to hide
        seed: Random seed for reproducible extraction
    
    Returns:
        Stego text with hidden message at random positions
    
    Note:
        The same seed must be used for extraction.
    """
    # YOUR CODE HERE (BONUS)
    zw_string = encode_to_zw(secret_message)
    n = len(cover_text)
    rng = random.Random(seed)
    
    # positions[i] is the index of i_th zw char in cover text, 
    # is also the number of visible chars appeared before this ZW char
    positions = [rng.randint(0, n) for _ in range(len(zw_string))]
    
    # Group ZW characters by their target insertion position
    # If multiple chars are in the same position, they will be kept in the order of their appearance
    groups = {}
    for pos, zw_char in zip(positions, zw_string):
        if pos not in groups:
            groups[pos] = []
        groups[pos].append(zw_char)
        
    stego_list = []
    for i in range(0, n + 1):
        # Insert ZW chars
        if i in groups:
            stego_list.extend(groups[i])
        # Insert original char
        if i < n:
            stego_list.append(cover_text[i])
            
    stego_text = ''.join(stego_list)
    return stego_text
    

def extract_random(stego_text: str, seed: int) -> str:
    """
    Extract a message embedded with random position distribution.
    
    Args:
        stego_text: Text containing hidden message
        seed: The same seed used during embedding
    
    Returns:
        Extracted secret message
    """
    # YOUR CODE HERE (BONUS)
    visible_chars = []
    zw_with_original_pos = []
    
    current_visible_idx = 0
    for char in stego_text:
        if char in ZW_SET:
            # ZW char with original inserted position
            zw_with_original_pos.append((current_visible_idx, char))
        else:
            visible_chars.append(char)
            current_visible_idx += 1
            
    if not zw_with_original_pos:
        return ''
    
    # Reconstruct the original cover text length and regenerate the same random positions embedded
    n = len(visible_chars)
    rng = random.Random(seed)
    positions = [rng.randint(0, n) for _ in range(len(zw_with_original_pos))]
    
    # expected_order[i] = (gen_idx, position)
    #    i    : the index of the ZW char as it appears left-to-right in stego text
    # gen_idx : which generation slot it belongs to in the original zw_string
    expected_order = sorted(enumerate(positions), key=lambda x: x[1])
    
    # Reconstruct zw_string in original generation order.
    # For each stego position i, expected_order tells us which gen_idx owns it,
    # so we place the actual ZW char from zw_with_original_pos[i] back into
    # the correct generation slot.
    decoded_zw = [None] * len(positions)
    for i, (gen_idx, _) in enumerate(expected_order):
        decoded_zw[gen_idx] = zw_with_original_pos[i][1]
        
    return decode_from_zw(''.join(decoded_zw))

In [15]:
# =============================================================================
# TEST: Random Position Embedding (BONUS)
# =============================================================================

# Uncomment to test after implementation
cover = "The quick brown fox jumps over the lazy dog"
secret = "Hidden!"
seed = 42

stego = embed_random(cover, secret, seed)
extracted = extract_random(stego, seed)
assert extracted == secret, f"Random extraction failed: got '{extracted}'"

# # Test with wrong seed
wrong_extracted = extract_random(stego, 123)
assert wrong_extracted != secret, "Wrong seed should not extract correctly"

print("✓ All random embedding tests passed!")

✓ All random embedding tests passed!


---
## Test Cases

The following test cases will be used for grading. Make sure your implementation passes all of them.

**Note:** You need to have the test files (`msg.txt`, `cover.txt`, `correct_stego.txt`, etc.) in the same directory as this notebook.

In [16]:
# =============================================================================
# OFFICIAL TEST 1: Embedding with Strategy 1
# =============================================================================

print("=" * 60)
print("OFFICIAL TEST 1: Embedding with Strategy 1 ")
print("=" * 60)

# Read inputs
with open('msg.txt', 'r', encoding='utf-8') as f:
    secret_message = f.read()
with open('cover.txt', 'r', encoding='utf-8') as f:
    cover_text = f.read()

# Embed
stego_text = embed(cover_text, secret_message, strategy=1)

# Write output
with open('stego_s1.txt', 'w', encoding='utf-8') as f:
    f.write(stego_text)

# Compare with correct output
with open('stego_s1.txt', 'r', encoding='utf-8') as f:
    student_stego = f.read()
with open('correct_stego_s1.txt', 'r', encoding='utf-8') as f:
    correct_stego = f.read()

assert student_stego == correct_stego, "Stego text does not match expected output for strategy 1"
print("Official Test 1 Passed! ✓")
print()

OFFICIAL TEST 1: Embedding with Strategy 1 
Official Test 1 Passed! ✓



In [17]:
# =============================================================================
# OFFICIAL TEST 2: Embedding with Strategy 2 
# =============================================================================

print("=" * 60)
print("OFFICIAL TEST 2: Embedding with Strategy 2 ")
print("=" * 60)

# Embed
stego_text = embed(cover_text, secret_message, strategy=2)

# Write output
with open('stego_s2.txt', 'w', encoding='utf-8') as f:
    f.write(stego_text)

# Compare with correct output
with open('stego_s2.txt', 'r', encoding='utf-8') as f:
    student_stego = f.read()
with open('correct_stego_s2.txt', 'r', encoding='utf-8') as f:
    correct_stego = f.read()

assert student_stego == correct_stego, "Stego text does not match expected output for strategy 2"
print("Official Test 2 Passed! ✓")
print()

OFFICIAL TEST 2: Embedding with Strategy 2 
Official Test 2 Passed! ✓



In [18]:
# =============================================================================
# OFFICIAL TEST 3: Embedding with Strategy 3 
# =============================================================================

print("=" * 60)
print("OFFICIAL TEST 3: Embedding with Strategy 3 ")
print("=" * 60)

# Embed
stego_text = embed(cover_text, secret_message, strategy=3)

# Write output
with open('stego_s3.txt', 'w', encoding='utf-8') as f:
    f.write(stego_text)

# Compare with correct output
with open('stego_s3.txt', 'r', encoding='utf-8') as f:
    student_stego = f.read()
with open('correct_stego_s3.txt', 'r', encoding='utf-8') as f:
    correct_stego = f.read()

assert student_stego == correct_stego, "Stego text does not match expected output for strategy 3"
print("Official Test 3 Passed! ✓")
print()

OFFICIAL TEST 3: Embedding with Strategy 3 
Official Test 3 Passed! ✓



In [19]:
# =============================================================================
# OFFICIAL TEST 4: Extraction from Strategy 1 
# =============================================================================

print("=" * 60)
print("OFFICIAL TEST 4: Extraction from Strategy 1 ")
print("=" * 60)

# Read stego file
with open('correct_stego_s1.txt', 'r', encoding='utf-8') as f:
    stego_text = f.read()

# Extract
extracted_message = extract(stego_text)

# Write output
with open('extracted_s1.txt', 'w', encoding='utf-8') as f:
    f.write(extracted_message)

# Compare with original message
with open('msg.txt', 'r', encoding='utf-8') as f:
    original_message = f.read()

assert extracted_message == original_message, f"Extracted message '{extracted_message}' does not match original '{original_message}'"
print(f"Extracted: '{extracted_message}'")
print("Official Test 4 Passed! ✓")
print()

OFFICIAL TEST 4: Extraction from Strategy 1 
Extracted: 'Be calm'
Official Test 4 Passed! ✓



In [20]:
# =============================================================================
# OFFICIAL TEST 5: Extraction from Strategy 2 
# =============================================================================

print("=" * 60)
print("OFFICIAL TEST 5: Extraction from Strategy 2 ")
print("=" * 60)

# Read stego file
with open('correct_stego_s2.txt', 'r', encoding='utf-8') as f:
    stego_text = f.read()

# Extract
extracted_message = extract(stego_text)

# Compare with original message
assert extracted_message == original_message, f"Extracted message '{extracted_message}' does not match original '{original_message}'"
print(f"Extracted: '{extracted_message}'")
print("Official Test 5 Passed! ✓")
print()

OFFICIAL TEST 5: Extraction from Strategy 2 
Extracted: 'Be calm'
Official Test 5 Passed! ✓



In [21]:
# =============================================================================
# OFFICIAL TEST 6: Extraction from Strategy 3 
# =============================================================================

print("=" * 60)
print("OFFICIAL TEST 6: Extraction from Strategy 3 ")
print("=" * 60)

# Read stego file
with open('correct_stego_s3.txt', 'r', encoding='utf-8') as f:
    stego_text = f.read()

# Extract
extracted_message = extract(stego_text)

# Compare with original message
assert extracted_message == original_message, f"Extracted message '{extracted_message}' does not match original '{original_message}'"
print(f"Extracted: '{extracted_message}'")
print("Official Test 6 Passed! ✓")
print()


OFFICIAL TEST 6: Extraction from Strategy 3 
Extracted: 'Be calm'
Official Test 6 Passed! ✓



In [22]:
# =============================================================================
# OFFICIAL TEST 7: Longer Message Embedding & Extraction 
# =============================================================================

print("=" * 60)
print("OFFICIAL TEST 7: Longer Message Embedding & Extraction ")
print("=" * 60)

# Read longer message
with open('msg2.txt', 'r', encoding='utf-8') as f:
    secret_message_2 = f.read()

# Embed
stego_text = embed(cover_text, secret_message_2, strategy=1)

# Write output
with open('stego_msg2.txt', 'w', encoding='utf-8') as f:
    f.write(stego_text)

# Compare with correct output
with open('stego_msg2.txt', 'r', encoding='utf-8') as f:
    student_stego = f.read()
with open('correct_stego_msg2.txt', 'r', encoding='utf-8') as f:
    correct_stego = f.read()

assert student_stego == correct_stego, "Stego text does not match expected output for msg2"

# Extract and verify
extracted = extract(stego_text)
assert extracted == secret_message_2, f"Extracted '{extracted}' does not match original '{secret_message_2}'"
print(f"Embedded and extracted: '{extracted}'")
print("Official Test 7 Passed! ✓")
print()

OFFICIAL TEST 7: Longer Message Embedding & Extraction 
Embedded and extracted: 'Be calm, Alice'
Official Test 7 Passed! ✓



In [23]:
# =============================================================================
# OFFICIAL TEST 8: Detection 
# =============================================================================

print("=" * 60)
print("OFFICIAL TEST 8: Detection ")
print("=" * 60)

# Test detection on clean text
clean_result = detect_zero_width(cover_text)
assert clean_result['has_zero_width'] == False, "Clean text should not have zero-width chars"
assert clean_result['total_zw_count'] == 0, "Clean text should have 0 zero-width chars"
print("Clean text detection: OK")

# Test detection on stego text
with open('correct_stego_s1.txt', 'r', encoding='utf-8') as f:
    stego_text = f.read()

stego_result = detect_zero_width(stego_text)
assert stego_result['has_zero_width'] == True, "Stego text should have zero-width chars"
assert stego_result['total_zw_count'] > 0, "Stego text should have positive zero-width count"
assert stego_result['suspicious'] == True, "Stego text should be flagged as suspicious"
print(f"Stego text detection: found {stego_result['total_zw_count']} ZW chars, suspicious={stego_result['suspicious']}")

print("Official Test 8 Passed! ✓")
print()

OFFICIAL TEST 8: Detection 
Clean text detection: OK
Stego text detection: found 36 ZW chars, suspicious=True
Official Test 8 Passed! ✓



---
## 11. Analysis Questions (1 points)

Answer the following questions about the zero width steganography method.

### Question 1 (0.25 points)

Why do we need a 16-bit length prefix in the message format? What problem would occur during extraction if we didn't include it?

**Answer**: The 16-length prefix tells decoder how many bits belong to the actual message. The decoder cannot distinguish real message bits from added padding bits (to make the total bits divisible by 2) without the 16-length prefix, so the decoded message can be wrong.

### Question 2 (0.25 points)

Why do we use 4 different zero-width characters instead of just 1? How does this affect the embedding capacity?

**Answer**: Using 4 different zero-width characters allows embedding 2 bits per 1 zero-width character, as $2^2 = 4$. If using only 1 zero-width character, we could embed 1 bit per character (bit 1 = embed, bit 0 = do not embed). With the same number of invisible characters, using 4 characters doubles the capacity, making the embedding more efficient and reducing the total zero-width character to embed.

### Question 3 (0.25 points)

Compare Strategy 1 (append at end) vs Strategy 3 (after each character) in terms of: (a) detection resistance, and (b) robustness to text editing.

**Answer**:

|| Strategy 1 | Strategy 3 |
|:---|:---|:---|
| Detection resistance | lower | higher |
| Robustness to text editing | higher | lower |

(a) Detection resistance:
- Strategy 1 is easier to detect because all zero-width characters are concentrated at the end of the text, a suspicious long sequences of only ZW chars.
- Strategy 3 distributes the hidden data uniformly after every visible character, making the distribution less obviously anomalous.

(b) Robustness to text editing: 
- Strategy 1 is generally more robust to text editing since the hidden data is stored at the end of cover text, as long as the editor does not delete or truncate the very end of the document, the entire hidden data survives.
- Strategy 3 is less robust because the hidden data is embedded throughout the entire text, even a small edit at any position in text can corrupt the hidden data.

### Question 4 (0.25 points)

Why does the extract() function work correctly regardless of which embedding strategy (1, 2, or 3) was used? What property of our encoding scheme makes this possible?

**Answer**: The `extract()` function works correctly regardless of the embedding strategy because all 3 strategies insert the zero-width characters into the cover text in the ***same order***. The only difference between the strategies is how these characters are distributed within the cover text. After encoding into a sequence of zero-width characters, each strategy simply embeds this sequence at different positions in the text (at the end, after words, or after each character). However, the relative order of these zero-width characters remains unchanged. The `extract()` function scans the entire stego text and collects all zero-width characters in the order they appear. Because their order is preserved regardless of the embedding strategy, the function can reconstruct the original encoded sequence and correctly decode the hidden message.